# Sklearn Pipelines — Interview Quick Review (Regression + Multiclass Classification)

Direct conversion of the two PDFs into an `.ipynb`, lightly structured for quick review.

## 0) Common Imports

In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [2]:
# Text processing


# import pandas as pd

# from sentence_transformers import SentenceTransformer
# EMB_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
# emb_model = SentenceTransformer(EMB_MODEL_NAME)
# # 1) Semantic embeddings -> append as emb_0 ... emb_d-1
# E = emb_model.encode(df["text"].fillna("").astype(str).tolist(), normalize_embeddings=True)
# df = df.join(pd.DataFrame(E, index=df.index, columns=[f"emb_{i}" for i in range(E.shape[1])]))

# from sklearn.feature_extraction.text import TfidfVectorizer
# # 2) TF-IDF -> append as tfidf_<token> (kept sparse to avoid blowing up RAM)
# tfidf = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=2)
# T = tfidf.fit_transform(df["text"].fillna("").astype(str))
# df = df.join(pd.DataFrame.sparse.from_spmatrix(T, index=df.index, columns=[f"tfidf_{t}" for t in tfidf.get_feature_names_out()]))

## 1) Regression Pipeline (RandomForestRegressor + OrdinalEncoder)

In [6]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

np.random.seed(42)
df = pd.DataFrame({
    'feature1': np.random.rand(100),
    'feature2': np.random.rand(100),
    'priority': np.random.choice(['Low', 'Med', 'High'], size=100),
    'risk_level': np.random.choice(['Low', 'Normal', 'High'], size=100),
    'target': np.random.rand(100) * 100
})

X = df.drop('target', axis=1)
y = df['target']

num_features = X.select_dtypes(include=['number']).columns.tolist() #include=['float64', 'int64']
cat_features = ['priority', 'risk_level']
category_orders = [['Low', 'Med', 'High'], ['Low', 'Normal', 'High']]

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        categories=category_orders,
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])

pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

param_grid = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [None, 10, 20],
    'regressor__min_samples_split': [2, 5]
}

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_grid,
    n_iter=5,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)

search.fit(X_train, y_train)
y_pred = search.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))
print("Best Parameters:", search.best_params_)

MAE: 27.19420942118698
R2 Score: -0.152767944147713
Best Parameters: {'regressor__n_estimators': 100, 'regressor__min_samples_split': 5, 'regressor__max_depth': 20}


## 2) Multiclass Classification Pipeline (RandomForestClassifier + OrdinalEncoder)

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.datasets import load_iris

data = load_iris(as_frame=True)
X = data.data
y = data.target

X['petal_order'] = pd.cut(
    X['petal length (cm)'],
    bins=[0, 2.5, 5, 7],
    labels=['Short', 'Medium', 'Long']
)
X['sepal_order'] = pd.cut(
    X['sepal width (cm)'],
    bins=[1, 2.5, 3.5, 5],
    labels=['Low', 'Med', 'High']
)

X = X.drop(columns=['petal length (cm)', 'sepal width (cm)'])

num_features = X.select_dtypes(include='number').columns.tolist()#include=['float64']
ord_features = ['petal_order', 'sepal_order']
categories_order = [['Short', 'Medium', 'Long'], ['Low', 'Med', 'High']]

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

ord_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        categories=categories_order,
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_features),
    ('ord', ord_pipeline, ord_features)
])

pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [None, 5, 10],
    'classifier__min_samples_split': [2, 5]
}

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_grid,
    n_iter=5,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42
)

search.fit(X_train, y_train)
y_pred = search.predict(X_test)

print(classification_report(y_test, y_pred))
print("Best parameters:", search.best_params_)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         9
           2       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30

Best parameters: {'classifier__n_estimators': 100, 'classifier__min_samples_split': 5, 'classifier__max_depth': 10}


In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer

embed_mod = SentenceTransformer("all-MiniLM-L6-v2")

emb = embed_mod.encode(df.text.tolist(),show_progress_bar = True)
emb.shape


import pandas as pd

# Define column names based on the width of the embeddings (384)
column_names = [f"emb_{i+1}" for i in range(emb.shape[1])]

# Create the DataFrame
# We don't need to specify 'index' unless you have specific row labels
df_emb = pd.DataFrame(data=emb, columns=column_names)

# Display the first few rows
df_emb.head()

from pycaret.classification import *

## 3) Quick swaps (optional)
- Use `OneHotEncoder(handle_unknown="ignore")` when category order doesn't matter.
- Swap estimators: `LogisticRegression`, `Ridge`, `XGBClassifier/Regressor` (if allowed).
- Change scoring: `f1_macro`, `roc_auc_ovr`, `neg_mean_squared_error`, etc.